# Direct Silver Crypto OHLC 1d

Heavy notebook implementation for the Silver market crypto OHLC path.

This notebook reads Bronze Coinbase daily OHLC data, reapplies key-level deduplication, parses `product_id` into `base_asset` and `quote_currency`, enforces Silver data quality rules, and merges the cleaned result into `slv_market.crypto_ohlc_1d`.

Execution modes:

- `backfill`: explicit inclusive date range
- `incremental`: rolling lookback window ending on the latest completed UTC day

The Bronze and Silver target tables must already exist. Run `00_platform_setup_catalog_schema.ipynb` first.

Default product set: `BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD`.

In [ ]:
import uuid
from datetime import date, datetime, timedelta, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("product_ids", "BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "3")
dbutils.widgets.text("run_id", str(uuid.uuid4()))

CATALOG = dbutils.widgets.get("catalog").strip() or "market_macro"
RAW_PRODUCT_IDS = dbutils.widgets.get("product_ids").strip()
MODE = dbutils.widgets.get("mode").strip().lower()
START_DATE_RAW = dbutils.widgets.get("start_date").strip()
END_DATE_RAW = dbutils.widgets.get("end_date").strip()
LOOKBACK_DAYS_RAW = dbutils.widgets.get("lookback_days").strip()
RUN_ID = dbutils.widgets.get("run_id").strip()

SOURCE_TABLE = f"{CATALOG}.brz_market.raw_coinbase_ohlc_1d"
TARGET_TABLE = f"{CATALOG}.slv_market.crypto_ohlc_1d"
UTC = timezone.utc


In [ ]:
def parse_product_ids(raw_value: str) -> list[str]:
    product_ids = [item.strip().upper() for item in raw_value.split(",") if item.strip()]
    product_ids = list(dict.fromkeys(product_ids))
    if not product_ids:
        raise ValueError("product_ids cannot be empty")
    return product_ids


def parse_iso_date(field_name: str, raw_value: str) -> date:
    try:
        return datetime.strptime(raw_value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(f"{field_name} must be in YYYY-MM-DD format") from exc


def resolve_date_window(
    mode: str,
    start_date_raw: str,
    end_date_raw: str,
    lookback_days_raw: str,
) -> tuple[date, date]:
    latest_complete_date = datetime.now(UTC).date() - timedelta(days=1)

    if mode not in {"backfill", "incremental"}:
        raise ValueError("mode must be either 'backfill' or 'incremental'")

    if mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("backfill mode requires both start_date and end_date")

        start_date = parse_iso_date("start_date", start_date_raw)
        end_date = parse_iso_date("end_date", end_date_raw)
    else:
        try:
            lookback_days = int(lookback_days_raw or "0")
        except ValueError as exc:
            raise ValueError("lookback_days must be an integer in incremental mode") from exc

        if lookback_days < 1:
            raise ValueError("lookback_days must be >= 1 in incremental mode")

        end_date = parse_iso_date("end_date", end_date_raw) if end_date_raw else latest_complete_date
        start_date = end_date - timedelta(days=lookback_days - 1)

    if start_date > end_date:
        raise ValueError("start_date cannot be after end_date")

    if end_date > latest_complete_date:
        raise ValueError(
            f"end_date must be <= latest completed UTC day: {latest_complete_date.isoformat()}"
        )

    return start_date, end_date


def collect_product_counts(df, count_alias: str) -> dict[str, int]:
    counts = (
        df.groupBy("product_id")
        .count()
        .withColumnRenamed("count", count_alias)
        .collect()
    )
    return {row["product_id"]: int(row[count_alias]) for row in counts}


In [ ]:
def run_silver_crypto_ohlc_1d() -> dict:
    product_ids = parse_product_ids(RAW_PRODUCT_IDS)
    start_date, end_date = resolve_date_window(MODE, START_DATE_RAW, END_DATE_RAW, LOOKBACK_DAYS_RAW)

    if not spark.catalog.tableExists(SOURCE_TABLE):
        raise RuntimeError(
            f"Source table {SOURCE_TABLE} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )

    if not spark.catalog.tableExists(TARGET_TABLE):
        raise RuntimeError(
            f"Target table {TARGET_TABLE} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )

    bronze_df = (
        spark.table(SOURCE_TABLE)
        .select(
            "product_id",
            "bar_date",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "source_window_start",
            "source_window_end",
            "ingested_at",
            "run_id",
            "payload_hash",
        )
        .filter(F.col("product_id").isin(product_ids))
        .filter((F.col("bar_date") >= F.lit(start_date)) & (F.col("bar_date") <= F.lit(end_date)))
    )

    rows_read = bronze_df.count()
    per_product_rows_read = collect_product_counts(bronze_df, "rows_read") if rows_read else {}

    if rows_read == 0:
        return {
            "status": "success_empty",
            "mode": MODE,
            "product_ids": product_ids,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_table": SOURCE_TABLE,
            "target_table": TARGET_TABLE,
            "rows_read": 0,
            "rows_after_dedup": 0,
            "rows_structural_invalid": 0,
            "rows_rejected": 0,
            "rows_to_update": 0,
            "rows_to_insert": 0,
            "rows_merged": 0,
            "run_id": RUN_ID,
            "per_product_rows_read": per_product_rows_read,
            "per_product_rows_after_dedup": {},
            "per_product_rows_rejected": {},
            "per_product_rows_merged": {},
        }

    dedup_window = Window.partitionBy("product_id", "bar_date").orderBy(
        F.col("source_window_end").desc(),
        F.col("ingested_at").desc(),
        F.col("payload_hash").desc(),
    )

    bronze_latest_df = (
        bronze_df
        .withColumn("_row_number", F.row_number().over(dedup_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    rows_after_dedup = bronze_latest_df.count()
    per_product_rows_after_dedup = collect_product_counts(bronze_latest_df, "rows_after_dedup")

    product_parts = F.split(F.col("product_id"), "-")
    structural_invalid_df = bronze_latest_df.filter(
        F.col("product_id").isNull()
        | F.col("bar_date").isNull()
        | (F.size(product_parts) != 2)
        | (F.element_at(product_parts, 1) == "")
        | (F.element_at(product_parts, 2) == "")
    )

    rows_structural_invalid = structural_invalid_df.count()
    if rows_structural_invalid > 0:
        display(structural_invalid_df.orderBy("product_id", "bar_date"))
        raise RuntimeError(
            f"Found {rows_structural_invalid} structurally invalid Bronze rows. Fix Bronze data before loading Silver."
        )

    silver_ingested_at = datetime.now(UTC)
    transformed_df = (
        bronze_latest_df
        .withColumn("base_asset", F.element_at(product_parts, 1))
        .withColumn("quote_currency", F.element_at(product_parts, 2))
    )

    dq_reason = (
        F.when(F.col("open").isNull(), F.lit("open_null"))
        .when(F.col("high").isNull(), F.lit("high_null"))
        .when(F.col("low").isNull(), F.lit("low_null"))
        .when(F.col("close").isNull(), F.lit("close_null"))
        .when(F.col("volume").isNull(), F.lit("volume_null"))
        .when(F.col("open") < 0, F.lit("open_negative"))
        .when(F.col("high") < 0, F.lit("high_negative"))
        .when(F.col("low") < 0, F.lit("low_negative"))
        .when(F.col("close") < 0, F.lit("close_negative"))
        .when(F.col("volume") < 0, F.lit("volume_negative"))
        .when(F.col("high") < F.col("low"), F.lit("high_below_low"))
        .when(F.col("open") < F.col("low"), F.lit("open_below_low"))
        .when(F.col("open") > F.col("high"), F.lit("open_above_high"))
        .when(F.col("close") < F.col("low"), F.lit("close_below_low"))
        .when(F.col("close") > F.col("high"), F.lit("close_above_high"))
    )

    assessed_df = transformed_df.withColumn("dq_reason", dq_reason)
    rejected_df = assessed_df.filter(F.col("dq_reason").isNotNull())
    rows_rejected = rejected_df.count()
    per_product_rows_rejected = collect_product_counts(rejected_df, "rows_rejected") if rows_rejected else {}

    valid_df = (
        assessed_df
        .filter(F.col("dq_reason").isNull())
        .select(
            "product_id",
            "bar_date",
            "base_asset",
            "quote_currency",
            "open",
            "high",
            "low",
            "close",
            "volume",
        )
        .withColumn("ingested_at", F.lit(silver_ingested_at))
        .withColumn("run_id", F.lit(RUN_ID))
    )

    rows_valid = valid_df.count()
    per_product_rows_merged = collect_product_counts(valid_df, "rows_merged") if rows_valid else {}

    if rows_valid == 0:
        if rows_rejected > 0:
            display(rejected_df.orderBy("product_id", "bar_date"))

        return {
            "status": "success_empty_valid",
            "mode": MODE,
            "product_ids": product_ids,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "source_table": SOURCE_TABLE,
            "target_table": TARGET_TABLE,
            "rows_read": rows_read,
            "rows_after_dedup": rows_after_dedup,
            "rows_structural_invalid": 0,
            "rows_rejected": rows_rejected,
            "rows_to_update": 0,
            "rows_to_insert": 0,
            "rows_merged": 0,
            "run_id": RUN_ID,
            "per_product_rows_read": per_product_rows_read,
            "per_product_rows_after_dedup": per_product_rows_after_dedup,
            "per_product_rows_rejected": per_product_rows_rejected,
            "per_product_rows_merged": {},
        }

    existing_key_count = (
        valid_df.select("product_id", "bar_date")
        .join(
            spark.table(TARGET_TABLE).select("product_id", "bar_date"),
            on=["product_id", "bar_date"],
            how="inner",
        )
        .count()
    )

    DeltaTable.forName(spark, TARGET_TABLE).alias("tgt").merge(
        valid_df.alias("src"),
        "tgt.product_id = src.product_id AND tgt.bar_date = src.bar_date",
    ).whenMatchedUpdate(
        set={
            "base_asset": "src.base_asset",
            "quote_currency": "src.quote_currency",
            "open": "src.open",
            "high": "src.high",
            "low": "src.low",
            "close": "src.close",
            "volume": "src.volume",
            "ingested_at": "src.ingested_at",
            "run_id": "src.run_id",
        }
    ).whenNotMatchedInsertAll().execute()

    display(valid_df.orderBy("product_id", "bar_date"))
    if rows_rejected > 0:
        display(rejected_df.orderBy("product_id", "bar_date"))

    return {
        "status": "success",
        "mode": MODE,
        "product_ids": product_ids,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "source_table": SOURCE_TABLE,
        "target_table": TARGET_TABLE,
        "rows_read": rows_read,
        "rows_after_dedup": rows_after_dedup,
        "rows_structural_invalid": 0,
        "rows_rejected": rows_rejected,
        "rows_to_update": existing_key_count,
        "rows_to_insert": rows_valid - existing_key_count,
        "rows_merged": rows_valid,
        "run_id": RUN_ID,
        "per_product_rows_read": per_product_rows_read,
        "per_product_rows_after_dedup": per_product_rows_after_dedup,
        "per_product_rows_rejected": per_product_rows_rejected,
        "per_product_rows_merged": per_product_rows_merged,
    }


result = run_silver_crypto_ohlc_1d()
result
